In [0]:
# =============================================================================
# Notebook  : 00_pipeline_run_logger
# Location  : /HGI-Lakehouse-Pipeline/00_Orchestration/00_pipeline_run_logger
# Purpose   : Shared utility. Every task in the master job calls this to:
#               1. Log run start / end / metrics into pipeline_run_log table
#               2. Print a rich summary report to the notebook output
#
# HOW TO USE from any notebook:
#   # At end of notebook, after processing:
#   dbutils.notebook.run(
#       "../00_Orchestration/00_pipeline_run_logger",
#       timeout_seconds=120,
#       arguments={
#           "customer_id":    customer_id,
#           "task_name":      "bronze_account",
#           "source_system":  "salesforce",
#           "object_name":    "account",
#           "layer":          "bronze",            # landing | bronze | silver_cdf | map
#           "status":         "success",           # success | failed | skipped | no_new_data
#           "records_before": str(records_before), # count before this run
#           "records_after":  str(records_after),  # count after this run
#           "files_found":    str(files_found),    # new files detected in landing zone
#           "error_message":  ""                   # empty string if no error
#       }
#   )
# =============================================================================

In [0]:
# CELL 1
%run ../config/pipeline_config

In [0]:
# CELL 2 ── Widgets (receive all metrics as strings — notebook.run passes strings)
dbutils.widgets.text("customer_id",    "cust_001")
dbutils.widgets.text("task_name",      "unknown")
dbutils.widgets.text("source_system",  "salesforce")
dbutils.widgets.text("object_name",    "unknown")
dbutils.widgets.text("layer",          "bronze")
dbutils.widgets.text("status",         "success")
dbutils.widgets.text("records_before", "0")
dbutils.widgets.text("records_after",  "0")
dbutils.widgets.text("files_found",    "0")
dbutils.widgets.text("error_message",  "")

customer_id    = dbutils.widgets.get("customer_id").strip().lower()
task_name      = dbutils.widgets.get("task_name").strip()
source_system  = dbutils.widgets.get("source_system").strip().lower()
object_name    = dbutils.widgets.get("object_name").strip().lower()
layer          = dbutils.widgets.get("layer").strip().lower()
status         = dbutils.widgets.get("status").strip().lower()
error_message  = dbutils.widgets.get("error_message").strip()

records_before = int(dbutils.widgets.get("records_before") or "0")
records_after  = int(dbutils.widgets.get("records_after")  or "0")
files_found    = int(dbutils.widgets.get("files_found")    or "0")

records_new    = max(records_after - records_before, 0)

from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

run_ts = datetime.utcnow()


In [0]:
# CELL 3 ── Create log table if not exists (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {meta_catalog}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {meta_catalog}.pipeline_run_log (
        run_id          STRING,
        customer_id     STRING,
        tenant_id       BIGINT,
        task_name       STRING,
        source_system   STRING,
        object_name     STRING,
        layer           STRING,
        status          STRING,
        records_before  BIGINT,
        records_after   BIGINT,
        records_new     BIGINT,
        files_found     BIGINT,
        run_timestamp   TIMESTAMP,
        error_message   STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

In [0]:
# CELL 4 ── Write this run's log entry
run_id = f"{customer_id}_{task_name}_{run_ts.strftime('%Y%m%d_%H%M%S')}"
try:
    tenant_id_val = int(customer_id.split("_")[1])
except Exception:
    tenant_id_val = -1

log_row = spark.createDataFrame([{
    "run_id":         run_id,
    "customer_id":    customer_id,
    "tenant_id":      tenant_id_val,
    "task_name":      task_name,
    "source_system":  source_system,
    "object_name":    object_name,
    "layer":          layer,
    "status":         status,
    "records_before": records_before,
    "records_after":  records_after,
    "records_new":    records_new,
    "files_found":    files_found,
    "run_timestamp":  run_ts,
    "error_message":  error_message,
}])

log_row.write.format("delta").mode("append").saveAsTable(f"{meta_catalog}.pipeline_run_log")

In [0]:
# CELL 5 ── Print formatted run report
STATUS_ICONS = {
    "success":      "✅",
    "failed":       "❌",
    "skipped":      "⏭",
    "no_new_data":  "⏸",
}
icon = STATUS_ICONS.get(status, "ℹ")

print("=" * 65)
print(f"  {icon}  PIPELINE RUN REPORT")
print("=" * 65)
print(f"  Run ID         : {run_id}")
print(f"  Customer       : {customer_id}  (tenant={tenant_id_val})")
print(f"  Task           : {task_name}")
print(f"  Layer          : {layer.upper()}")
print(f"  Source         : {source_system} / {object_name}")
print(f"  Status         : {status.upper()}")
print(f"  Timestamp      : {run_ts.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("-" * 65)
print(f"  Files detected : {files_found:>10,}")
print(f"  Records before : {records_before:>10,}")
print(f"  Records after  : {records_after:>10,}")
print(f"  Records NEW    : {records_new:>10,}  ← net new rows written this run")
if records_before > 0:
    pct = round(100 * records_new / records_before, 2)
    print(f"  Growth %       : {pct:>9.2f}%")
if error_message:
    print(f"  Error          : {error_message}")
print("=" * 65)


In [0]:
# CELL 6 ── Show last 10 runs for this customer (full history)
print(f"\n  ── Last 10 runs for {customer_id} ──")
spark.sql(f"""
    SELECT
        run_timestamp,
        task_name,
        layer,
        status,
        files_found,
        records_before,
        records_after,
        records_new,
        ROUND(CASE WHEN records_before > 0
              THEN 100.0 * records_new / records_before
              ELSE 0 END, 1) AS growth_pct
    FROM {meta_catalog}.pipeline_run_log
    WHERE customer_id = '{customer_id}'
    ORDER BY run_timestamp DESC
    LIMIT 10
""").show(truncate=False)

dbutils.notebook.exit("logged")
